# Adaptive tuning — mock agents or Groq
This notebook loads an Adaptive artifact, selects one test, and runs
**Designer → initial simulation → Tuner → final simulation**.

Run the cells from top to bottom using this repository's `.venv` Python kernel.
The API key is read only from `GROQ_API_KEY` in the environment or the repository-root `.env` file.
`USE_MOCK_AGENTS=True` uses mock agents without API calls. Set it to `False` for Groq; real runs may incur costs.

The example selects `single_pole.json / test_1`.
The pole starts at `[0.05, 0.0]` and tracks the `theta = 0.0` reference. Zero angle is upright; angles are in radians. The selected test includes small uncertainty and disturbance terms; the artifact file is not modified.


In [ ]:
import copy
import json
import math
import os
import sys
from contextlib import contextmanager
from datetime import datetime, timezone
from pathlib import Path
from pprint import pprint

import matplotlib.pyplot as plt
import numpy as np
from dotenv import load_dotenv

# Find the repo whether the kernel starts in this folder or above the repo.
REPO_ROOT_HINT = None  # Optionally set an absolute repository path.
search_start = Path(REPO_ROOT_HINT).resolve() if REPO_ROOT_HINT else Path.cwd().resolve()
candidates = []
for parent in (search_start, *search_start.parents):
    candidates.extend([parent, parent / "LabCD_NewModules-master"])
REPO_ROOT = next(
    (p for p in candidates if (p / "backend_core" / "AgentAdaptive").is_dir()
     and (p / "benchmarks" / "Adaptive" / "artifacts").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Set REPO_ROOT_HINT to the absolute repository path.")
for import_root in (REPO_ROOT, REPO_ROOT / "packages" / "labcd_agents" / "src"):
    if str(import_root) not in sys.path:
        sys.path.insert(0, str(import_root))

ADAPTIVE_DIR = REPO_ROOT / "benchmarks" / "Adaptive"
ARTIFACTS_DIR = ADAPTIVE_DIR / "artifacts"
RESULTS_DIR = ADAPTIVE_DIR / "json_reports" / "results" / "groq-gpt-oss-120b"
print("Repository:", REPO_ROOT)


## 1. Test selection and editable settings
`RUN_TUNING=True` starts a new run; `False` only loads and plots a saved report.
`ENABLE_TUNING=False` skips only the Tuner loop; with real agents, Designer and Reporter still make API calls.

The test directly specifies its initial state, output references, uncertainty, and disturbance. Adaptive has no numbered scenario-level preset.
The selected test already includes small uncertainty and disturbance terms, so the Adaptive estimators are enabled. The example-effect switches below optionally replace those terms.

`USE_MOCK_AGENTS` and `MOCK_SEED` are local test controls. Mock gains vary reproducibly; performance improvements are not guaranteed. Mock usage is marked `simulated`, with a synthetic $0.001/token cost and zero actual API cost. The confirmed artifact skips Clarifier chat in both modes.


In [ ]:
ARTIFACT_FILE = "single_pole.json"
TEST_ID = "test_1"
USE_MOCK_AGENTS = True  # False uses real Groq agents.
MOCK_SEED = 42          # Repeatable mock proposals and synthetic token counts.
RUN_TUNING = True
ENABLE_TUNING = True

LLM_PROVIDER = "groq"
ROLE_MODELS = {
    "design": "openai/gpt-oss-120b",
    "tuner": "openai/gpt-oss-120b",
    "clarifier": "openai/gpt-oss-120b",
    "reporter": "openai/gpt-oss-120b",
}
ROLE_MAX_TOKENS = {"design": 2000, "tuner": 2000, "clarifier": 8000, "reporter": 500}
API_KEY_ENV_VAR = "GROQ_API_KEY"
DOTENV_PATH = REPO_ROOT / ".env"

# Applied to the selected test in memory. These expressions are for this plant.
ADD_EXAMPLE_UNCERTAINTY = False
ADD_EXAMPLE_DISTURBANCE = False
EXAMPLE_UNCERTAINTY = [{"state": "omega", "expr": "-0.02 * omega"}]
EXAMPLE_DISTURBANCE = [{"state": "omega", "expr": "0.01 * sin(0.8*t)"}]

SIMULATION_TIME_S = 8.0
SOLVER_STEP_S = 0.001
TARGET_STEADY_RMS_FRACTION = 0.001
MAX_TUNING_ROUNDS = 4
TUNING_OBJECTIVES = {
    "steady_state_error": 5,
    "tracking_mse": 5,
    "transient_error": 3,
    "control_effort": 2,
}
SAVE_TIME_SERIES = False  # Kept in memory for plotting even when not saved.
EXPORT_MAX_POINTS = 2000
RESULTS_DIR = ADAPTIVE_DIR / "json_reports" / "results" / (
    "mock" if USE_MOCK_AGENTS else "groq-gpt-oss-120b"
)
RESULT_FILE = RESULTS_DIR / "batch_tuning_results.json"

DESCRIPTION = "Design and tune the single-pole test with zero angle upright and small uncertainty and disturbance."
CLARIFICATION_RECORD = []
CLARIFIER_USAGE = None  # Confirmed spec is supplied directly; no Clarifier chat.
SIMULATION_OVERRIDES = {"t_end": SIMULATION_TIME_S, "dt": SOLVER_STEP_S}


## 2. Build the applied controller specification
The selected test's initial condition replaces `dynamics.x0`. Its `references` define targets only for the declared `outputs`, not for every state.
Benchmark metadata, bounds, and complexity are not passed to the controller. Bounds are not enforced constraints in this simulator.


In [ ]:
from backend_core.AgentAdaptive.tools import system_spec as spec_tools

artifact_path = ARTIFACTS_DIR / ARTIFACT_FILE
artifact = json.loads(artifact_path.read_text(encoding="utf-8"))
combinations = artifact.get("tests", {}).get("combinations", [])
selected_tests = [test for test in combinations if test.get("test_id") == TEST_ID]
if len(selected_tests) != 1:
    available = [test.get("test_id") for test in combinations]
    raise ValueError(f"Expected one matching test; available IDs: {available}")
selected_test = copy.deepcopy(selected_tests[0])

# Convert the Plant Agent output without adding fields to the artifact.
from backend_core.plant_compiler import PlantCompiler

compiler = PlantCompiler()
validation = compiler.validate(artifact)
if not validation.ok:
    raise ValueError(f"Invalid Plant Agent artifact: {validation.errors}")
applied_spec = compiler.generate_adaptive_spec(
    plant_output=artifact,
    pre_launch={
        "initial_state": selected_test["initial_condition"],
        "total_simulation_time": SIMULATION_TIME_S,
        "solver_sample_time": SOLVER_STEP_S,
    },
)
dyn = applied_spec["dynamics"]
dyn["uncertainty"] = copy.deepcopy(selected_test["uncertainty"])
dyn["disturbance"] = copy.deepcopy(selected_test["disturbance"])
states = dyn["states"]
outputs = dyn["outputs"]
initial_state = np.asarray(selected_test["initial_condition"], dtype=float)
if initial_state.shape != (len(states),) or not np.all(np.isfinite(initial_state)):
    raise ValueError("IC must have one finite number per state.")
references = copy.deepcopy(selected_test["references"])
if not isinstance(references, list) or len(references) != len(outputs):
    raise ValueError("Provide one reference for each declared output.")
if (any(not isinstance(ref, dict) or not isinstance(ref.get("expr"), str)
        or not ref["expr"].strip() for ref in references)
        or {ref.get("output") for ref in references} != set(outputs)):
    raise ValueError("References must name each declared output exactly once with a nonempty expression.")
dyn["x0"] = initial_state.tolist()
dyn["references"] = references
dyn["sim_time"] = SIMULATION_TIME_S
dyn["solver_step"] = SOLVER_STEP_S
if ADD_EXAMPLE_UNCERTAINTY:
    dyn["uncertainty"] = copy.deepcopy(EXAMPLE_UNCERTAINTY)
if ADD_EXAMPLE_DISTURBANCE:
    dyn["disturbance"] = copy.deepcopy(EXAMPLE_DISTURBANCE)

if not (0 < SOLVER_STEP_S <= SIMULATION_TIME_S / 100) or MAX_TUNING_ROUNDS < 1:
    raise ValueError("Use a positive time step, at least 100 simulation steps, and at least one allowed tuning round.")
if not math.isfinite(SIMULATION_TIME_S) or not math.isfinite(SOLVER_STEP_S):
    raise ValueError("Simulation timing must be finite.")
if not math.isfinite(TARGET_STEADY_RMS_FRACTION) or TARGET_STEADY_RMS_FRACTION <= 0:
    raise ValueError("The target RMS fraction must be finite and positive.")
gaps = spec_tools.missing_items(applied_spec)
if gaps:
    raise ValueError(f"Incomplete Adaptive system_spec: {gaps}")

print("Plant:", applied_spec["system_name"], "| Test:", TEST_ID)
print("IC:", dyn["x0"], "| Output references:", dyn["references"])
print("Tracked outputs:", outputs)
print("Adaptive estimator enabled:", bool(dyn["uncertainty"] or dyn["disturbance"]))
pprint(applied_spec)


## 3. Execution configuration and report helpers
The module internally uses `OPENAI_*` environment-variable names, but the provider is explicitly set to Groq for every role.
These settings are temporary: the previous environment settings are restored after execution.
The API key is never printed or saved in JSON reports.


In [ ]:
@contextmanager
def groq_configuration():
    environment = {
        "LABCD_ADAPTIVE_SHOW_PLOTS": "0",
        "LABCD_ADAPTIVE_SERIES_MAX_POINTS": str(EXPORT_MAX_POINTS),
    }
    if not USE_MOCK_AGENTS:
        load_dotenv(dotenv_path=DOTENV_PATH, override=False)
        api_key = os.getenv(API_KEY_ENV_VAR)
        if not api_key:
            raise RuntimeError(f"Set {API_KEY_ENV_VAR} in the environment or {DOTENV_PATH}.")
        for role in ROLE_MODELS:
            suffix = "" if role == "design" else "_" + role.upper()
            environment["LLM_PROVIDER" + suffix] = LLM_PROVIDER
            environment["OPENAI_MODEL" + suffix] = ROLE_MODELS[role]
            environment["OPENAI_API_KEY" + suffix] = api_key
            environment["OPENAI_MAX_TOKENS" + suffix] = str(ROLE_MAX_TOKENS[role])
    previous = {key: os.environ.get(key) for key in environment}
    try:
        os.environ.update(environment)
        yield
    finally:
        for key, value in previous.items():
            if value is None:
                os.environ.pop(key, None)
            else:
                os.environ[key] = value

def to_jsonable(value):
    if isinstance(value, np.ndarray):
        return to_jsonable(value.tolist())
    if isinstance(value, np.generic):
        return to_jsonable(value.item())
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if value is None or isinstance(value, (str, int, bool)):
        return value
    if isinstance(value, dict):
        return {str(key): to_jsonable(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_jsonable(item) for item in value]
    if isinstance(value, Path):
        return str(value)
    raise TypeError(f"Unsupported result type: {type(value).__name__}")

progress_events = []
STOP_REQUESTED = False

def on_event(event):
    # The backend passes ONE event dictionary, not keyword arguments.
    progress_events.append(copy.deepcopy(event))
    if event.get("kind") in {"stage_start", "stage_done", "cancelled"}:
        print(f"[{event.get('stage')}] {event.get('kind')} | round={event.get('round', '-')}")
    elif event.get("kind") == "note":
        print(event.get("text", ""))

def should_stop():
    return STOP_REQUESTED


## 4. Execute with mock agents or Groq
`run_full_pipeline` derives the control law numerically/symbolically and runs the simulations.
The Designer selects a supported method from the pole's symbolic plant structure.
Initial gains come from the backend's configuration; the public function has no separate seed-gains argument.

If the initial design already meets the target error, the Tuner may skip additional rounds.
`RUN_TUNING=False` requires no API key or API calls, but a matching saved report must already exist.

The import selects the implementation: `tuner_agent.py` uses mock agents; `tuner_agent_main.py` uses the original LLM agents. Real agent files keep the `_main` suffix.


In [ ]:
result = None
usage = None
tuning_log = None
tuning_best = None
report = None
series = None

if RUN_TUNING:
    progress_events.clear()
    with groq_configuration():
        if USE_MOCK_AGENTS:
            from backend_core.AgentAdaptive.agents.tuner_agent import run_full_pipeline
            agent_options = {"mock_seed": MOCK_SEED}
        else:
            from backend_core.AgentAdaptive.agents.tuner_agent_main import run_full_pipeline
            agent_options = {}

        result, usage, tuning_log, tuning_best = run_full_pipeline(
            description=DESCRIPTION,
            enable_tuning=ENABLE_TUNING,
            target_rms_frac=TARGET_STEADY_RMS_FRACTION,
            max_tuning_rounds=MAX_TUNING_ROUNDS,
            on_event=on_event,
            should_stop=should_stop,
            clarification_record=CLARIFICATION_RECORD,
            sim_overrides=SIMULATION_OVERRIDES,
            clarifier_usage=CLARIFIER_USAGE,
            tuning_objectives=TUNING_OBJECTIVES,
            system_spec=applied_spec,
            **agent_options,
        )
    series = result.get("series")
    final_metrics = result.get("final_metrics")
    messages = result.get("messages") or []
    final_summary = getattr(messages[-1], "content", "") if messages else ""
    method_event = next(
        (event for event in progress_events if event.get("stage") == "design"
         and event.get("kind") == "stage_done"), {}
    )
    report = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "artifact": ARTIFACT_FILE,
        "plant": applied_spec["system_name"],
        "test_id": TEST_ID,
        "initial_condition": initial_state.tolist(),
        "run_configuration": {
            "provider": "mock" if USE_MOCK_AGENTS else LLM_PROVIDER,
            "models": usage.get("models", {}),
            "max_tokens": None if USE_MOCK_AGENTS else ROLE_MAX_TOKENS,
            "enable_tuning": ENABLE_TUNING,
            "target_steady_rms_fraction": TARGET_STEADY_RMS_FRACTION,
            "max_tuning_rounds": MAX_TUNING_ROUNDS,
            "objectives": TUNING_OBJECTIVES,
            "simulation_overrides": SIMULATION_OVERRIDES,
            "added_example_uncertainty": ADD_EXAMPLE_UNCERTAINTY,
            "added_example_disturbance": ADD_EXAMPLE_DISTURBANCE,
            "save_time_series": SAVE_TIME_SERIES,
            "export_max_points": EXPORT_MAX_POINTS,
        },
        "applied_system_spec": applied_spec,
        "agent_mode": "mock" if USE_MOCK_AGENTS else "groq",
        "mock_seed": MOCK_SEED if USE_MOCK_AGENTS else None,
        "method": method_event.get("args", {}).get("method"),
        "pipeline_completed": final_metrics is not None and series is not None,
        "tuning": {
            "mock_agent_messages": result.get("mock_agent_messages", []),
            "best_params": (tuning_best or {}).get("tuning"),
            "best_round": (tuning_best or {}).get("round"),
            "best_metrics": (tuning_best or {}).get("metrics"),
            "history": tuning_log,
            "token_usage": usage,
        },
        "verification": final_metrics,
        "final_summary": final_summary,
        "abstract": result.get("abstract"),
    }
    if SAVE_TIME_SERIES:
        report["series"] = series
else:
    if not RESULT_FILE.is_file():
        raise FileNotFoundError(f"No saved report at {RESULT_FILE}. Run once with RUN_TUNING=True.")
    report = json.loads(RESULT_FILE.read_text(encoding="utf-8"))
    saved_spec = report.get("applied_system_spec")
    if report.get("artifact") != ARTIFACT_FILE or report.get("test_id") != TEST_ID or saved_spec != applied_spec or report.get("agent_mode") != ("mock" if USE_MOCK_AGENTS else "groq"):
        raise ValueError("Saved report belongs to different applied test conditions; select its settings or tune again.")
    series = report.get("series")
    print("Loaded saved report; no API calls:", RESULT_FILE)


## 5. Save and inspect the result
The report contains the applied specification, best parameters, tuning history, token usage, and final simulation metrics.
Reports are saved under `Adaptive/json_reports/results/groq-gpt-oss-120b/`.
Each run updates `batch_tuning_results.json` and `batch_tuning_results_brief.json` in the selected model folder. A new run for that model replaces the previous pair.
Pipeline completion is different from tracking success (`verification.success`).

Mock reports go under `Adaptive/json_reports/results/mock/` and include the mock messages. Actual verification metrics come from the final controller simulation.


In [ ]:
if RUN_TUNING:
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    RESULT_FILE.write_text(
        json.dumps(to_jsonable(report), indent=2, ensure_ascii=False, allow_nan=False) + "\n",
        encoding="utf-8",
    )
    print("Saved JSON:", RESULT_FILE)

    from benchmarks.Adaptive.result_brief import write_brief_result

    brief_destination = write_brief_result(RESULT_FILE)
    print("Saved brief JSON:", brief_destination)

print("Selected method:", report.get("method"))
print("Pipeline completed:", report.get("pipeline_completed"))
print("Best tuning parameters:")
pprint(report.get("tuning", {}).get("best_params"))
print("Final simulation metrics:")
pprint(report.get("verification"))
print("Token totals:")
pprint((report.get("tuning", {}).get("token_usage") or {}).get("total"))

if not report.get("pipeline_completed"):
    raise RuntimeError("Adaptive design/simulation did not complete. Diagnostic saved: " + report.get("final_summary", ""))


## 6. Response and tuning-progress plots
The time-domain plot uses the final simulation. Exported series may be downsampled to `EXPORT_MAX_POINTS` samples.
Metrics are computed using all original simulation samples.
Set `SAVE_TIME_SERIES=True` during tuning to retain time-domain plots for later report-only runs.
With the default `False`, JSON reports stay compact and time-domain series are available only during the current tuning run.


In [ ]:
if series:
    channels = series["channels"]
    times = np.asarray(channels["t"]["data"], dtype=float)
    actual = np.asarray(channels["y"]["data"], dtype=float)
    reference = np.asarray(channels["ref"]["data"], dtype=float)
    controls = np.asarray(channels["u"]["data"], dtype=float)
    fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True, constrained_layout=True)
    for index, name in enumerate(channels["y"]["names"]):
        axes[0].plot(times, actual[:, index], label=f"{name}: actual", linewidth=1.8)
        axes[0].plot(times, reference[:, index], "--", label=f"{name}: reference", linewidth=1.5)
    for index, name in enumerate(channels["u"]["names"]):
        axes[1].plot(times, controls[:, index], label=name, linewidth=1.5)
    axes[0].set_title(f"{report['plant']} | {report['test_id']} | {report.get('method')}")
    axes[0].set_ylabel("Tracked output")
    axes[1].set_ylabel("Control input")
    axes[1].set_xlabel("Time (s)")
    for axis in axes:
        axis.grid(alpha=0.25)
        axis.legend()
    plt.show()
else:
    print("No saved time series; showing the saved tuning history only.")

history = report.get("tuning", {}).get("history") or []
valid_rounds = [
    entry for entry in history
    if isinstance(entry.get("tracking_pct_headline"), (int, float))
]
if valid_rounds:
    fig, axis = plt.subplots(figsize=(9, 3.5), constrained_layout=True)
    rounds = [entry["round"] for entry in valid_rounds]
    axis.plot(rounds, [entry["tracking_pct_headline"] for entry in valid_rounds], "o-", linewidth=1.8)
    axis.set_xticks(rounds)
    axis.set_ylim(0, 105)
    axis.set_xlabel("Tuning round (0 = initial design)")
    axis.set_ylabel("Worst-output steady tracking (%)")
    axis.set_title("Adaptive tracking score — not the MPC exponential success score")
    axis.grid(alpha=0.25)
    plt.show()
